In [ ]:
import pathlib

import altair as alt
import geopandas as gpd
import laspy
import logging
import numpy as np
import open3d as o3d
import pyproj
import shapely 

import pitchmark
import pitchmark.osm
import pitchmark.lidar

In [ ]:
logging.basicConfig(level=logging.INFO)

In [ ]:
logging.getLogger()

In [ ]:
alt.renderers.enable('mimetype')
alt.renderers

In [ ]:
logging.warning("abc")
logging.info("def")

In [ ]:
pathlib.Path().cwd()

In [ ]:
pathlib.Path().cwd().parent

In [ ]:
map_path = pathlib.Path().cwd().parent / "data-raw" / "OpenStreetMap" / "augusta_national" / "map.osm"
handler = pitchmark.osm.GolfHandler()
handler.apply_file(map_path)
fc = handler.feature_collection
augusta_national = pitchmark.Course.from_featurecollection(fc)

In [ ]:
geoseries = augusta_national.gdf.geometry
geoseries

In [ ]:
azalea = augusta_national.holes[12]
azalea

In [ ]:
redbud = augusta_national.holes[15]
redbud

In [ ]:
folder = pathlib.Path().cwd().parent / "data-raw" / "USGS_LIDAR"
files = [
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1284n1253.laz",
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1284n1254.laz",
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1285n1253.laz",
    "USGS_LPC_GA_Statewide_2018_B18_DRRA_e1285n1254.laz",
]
paths = [folder / file for file in files]
paths

In [ ]:
# las_classes = [
#     pitchmark.lidar.las.LasClassification.unclassified,
#     pitchmark.lidar.las.LasClassification.ground,
# ]
# pitchmark.lidar.las.clip_to_geoseries(
#     paths,
#     geoseries,
#     classification_filter=las_classes,
#     to_file=folder / "augusta_national_unclassified_or_ground.laz",
# )
ground = laspy.read(folder / "augusta_national_ground.laz")

In [ ]:
augusta_national.populate_hole_meshes(folder / "augusta_national_ground.laz", smooth_iters=10)

In [ ]:
redbud = augusta_national.holes[15]
redbud

In [ ]:
redbud_flag = augusta_national.transformer_to_local.transform(*redbud.path.coords[-1])
redbud_flag

In [ ]:
redbud_green = redbud.gdf[
    redbud.gdf.contains(shapely.Point(redbud_flag))
    & (redbud.gdf["course_area"] == "putting_green")
].unary_union
redbud_green_surround = redbud_green.buffer(5.0)
shapely.prepare(redbud_green_surround)
redbud_green_surround


In [ ]:
surround_features = pitchmark.plotting.chart_course(redbud.gdf.clip(redbud_green_surround))

In [ ]:
selected_mesh = redbud.mesh[redbud.mesh.within(redbud_green)]
grades = (alt.Chart(selected_mesh)
    .mark_geoshape(
        filled=True,
        color="lightgray",
    )
    .encode(
        color=alt.Color("slope_grade:Q", scale=alt.Scale(domain=[0, 12.0], scheme="greys"))
    )
    .project(
        type="identity",
        reflectY=True,
    )
)

In [ ]:
selector = alt.selection_single(on="mouseover", nearest=True)
inclines = (
    alt.Chart(selected_mesh)
    .mark_point(
        shape="wedge",
        filled=True,
    )
    .encode(
        longitude="x",
        latitude="y",
        color=alt.condition(
            selector,
            alt.value("red"),
            alt.value("black")
        ),
        angle="slope_heading",
        size="slope_grade",
        tooltip=["z", "slope_heading", "slope_grade"],
    )
    .add_selection(selector)
    .project(
        type="identity",
        reflectY=True,
    )
    .properties(
        width=800,
        height=800,
    )
)


In [ ]:
surround_features + grades + inclines

In [ ]:
xmin, ymin, xmax, ymax = redbud_green.bounds
xmin, ymin, xmax, ymax

In [ ]:
cell_size = 1.0
nx = int((xmax - xmin) / cell_size)
np.linspace(xmin, xmax, nx)

In [ ]:
xrange, xadd = divmod(xmax - xmin, cell_size)
yrange, yadd = divmod(ymax - ymin, cell_size)

In [ ]:
# Square grid
# xspace = cell_size
# yspace = cell_size
# xs = np.arange(xmin + xadd/2, xmax, xspace)
# ys = np.arange(ymin + yadd/2, ymax, yspace)
# Hex grid is equal to two overlapping rectangular grids
xspace = cell_size
yspace = cell_size * np.sqrt(3)
xs1 = np.arange(xmin + xadd/2, xmax, xspace)
ys1 = np.arange(ymin + yadd/2, ymax, yspace)
grid1 = [shapely.Point(x, y) for x in xs1 for y in ys1]

In [ ]:
xs2 = np.arange(xmin + xadd/2 - xspace/2, xmax, xspace)
ys2 = np.arange(ymin + yadd/2 + yspace/2, ymax, yspace)
grid2 = [shapely.Point(x, y) for x in xs2 for y in ys2]
grid_points = grid1 + grid2

In [ ]:
len(grid_points)

In [ ]:
gdf = gpd.sjoin(
    gpd.GeoDataFrame(geometry=grid_points, crs=redbud.mesh.crs),
    redbud.mesh,
    how="left",
    predicate="within"
).clip(redbud_green)

In [ ]:
len(gdf)

In [ ]:
gdf["lat"] = gdf.geometry.x
gdf["lon"] = gdf.geometry.y
gdf

In [ ]:
selector = alt.selection_single(on="mouseover", nearest=True)
inclines = (
    alt.Chart(gdf)
    .mark_point(
        shape="wedge",
        filled=True,
        color="lightgray",
    )
    .encode(
        longitude="lat",
        latitude="lon",
        color=alt.condition(
            selector,
            alt.value("red"),
            alt.Color("slope_grade:Q", scale=alt.Scale(domainMin=0.0, scheme="greys")),
        ),
        angle="slope_heading",
        size="slope_grade",
        tooltip=["x", "y", "z", "slope_heading", "slope_grade"],
    )
    .add_selection(selector)
    .project(
        type="identity",
        reflectY=True,
    )
    .properties(
        width=600,
        height=600,
    )
)


In [ ]:
surround_features = pitchmark.plotting.chart_course(
    redbud.gdf.clip(redbud_green_surround)
).project(
    type="identity",
    reflectY=True,
)


In [ ]:
(surround_features + inclines)

In [ ]:
# Need to check that Taubin smoothening does not rotate the mesh so gravity is no longer pointing down!

In [ ]:
np.tan(np.deg2rad(2))

In [ ]:
redbud_green.area*9

In [ ]:
triangle_mesh = pitchmark.geom.simplified_mesh(redbud.mesh.geometry)

In [ ]:
np.asarray(triangle_mesh.vertices)

In [ ]:
ground.header.parse_crs()

In [ ]:
augusta_national.gdf.crs

In [ ]:
augusta_national.gdf

In [ ]:
dir(gpd.io.file)

In [ ]:
?gpd.io.file.to_file

In [ ]:
redbud.gdf.to_csv(folder / "redbud_gdf.csv")

In [ ]:
pd.from_csv(folder / "redbud_gdf.csv")

In [ ]:
redbud_df_from_csv = pd.read_csv(folder / "redbud_gdf.csv", index_col=0)
geometry = gpd.GeoSeries.from_wkt(redbud_df_from_csv["geometry"])
gdf = gpd.GeoDataFrame(data=redbud_df_from_csv, geometry=geometry)

In [ ]:
redbud.mesh.to_csv(folder / "redbud_mesh.csv")